In [3]:
import os
import shutil
import warnings
import pandas as pd
import geopandas as gpd

# ==========================================
#  MOUNT GOOGLE DRIVE SAFELY
# ==========================================
try:
    from google.colab import drive


    if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
        print("🧹 Cleaning up fake local drive folder to allow secure mount...")
        shutil.rmtree('/content/drive')

    drive.mount('/content/drive', force_remount=True)
except ImportError:
    pass


warnings.filterwarnings('ignore')

import osmnx as ox

# ==========================================
#  OSMnx POI EXTRACTION
# ==========================================
print(" Initiating OSMnx Production Ingestion Pipeline...")

provinces = [
    "Western Province, Sri Lanka",
    "Central Province, Sri Lanka",
    "Southern Province, Sri Lanka",
    "North Western Province, Sri Lanka"
]

tags = {
    'amenity': ['school', 'hospital', 'bus_station', 'marketplace', 'university', 'college', 'bank', 'pharmacy'],
    'shop': ['supermarket', 'convenience', 'wholesale', 'mall']
}

all_partitions = []

for province in provinces:
    print(f"⏳ Securely downloading features for {province}...")
    try:
        # Fetch geographic features from OSMnx
        gdf = ox.features_from_place(province, tags=tags)
        gdf = gdf.reset_index()

        # Merge 'amenity' and 'shop' tags into a single 'POI_Type' column
        if 'amenity' in gdf.columns and 'shop' in gdf.columns:
            gdf['POI_Type'] = gdf['amenity'].fillna(gdf['shop'])
        elif 'amenity' in gdf.columns:
            gdf['POI_Type'] = gdf['amenity']
        elif 'shop' in gdf.columns:
            gdf['POI_Type'] = gdf['shop']
        else:
            gdf['POI_Type'] = 'Other'

        # Ensure we only keep POIs that have actual names
        if 'name' in gdf.columns:
            gdf = gdf.dropna(subset=['name'])
            gdf = gdf.rename(columns={'name': 'Name'})
        else:
            gdf['Name'] = 'Unknown'

        # Convert all large polygons/buildings to exact center points
        gdf['geometry'] = gdf['geometry'].centroid
        gdf['Latitude'] = gdf.geometry.y
        gdf['Longitude'] = gdf.geometry.x

        # Filter down to the Lakehouse schema requirements
        clean_df = gdf[['POI_Type', 'Name', 'Latitude', 'Longitude']].copy()
        clean_df['Province'] = province.split(",")[0]

        all_partitions.append(clean_df)
        print(f"✅ Extracted {len(clean_df)} POIs from {province}.")

    except Exception as e:
        print(f"❌ Error extracting {province}: {e}")

# ==========================================
# SAVE TO BRONZE LAKEHOUSE LAYER
# ==========================================
output_dir = '/content/drive/MyDrive/DataStorm_TeamZypher/lakehouse/bronze'
os.makedirs(output_dir, exist_ok=True)

if all_partitions:
    final_df = pd.concat(all_partitions, ignore_index=True)

    # Drop any edge cases where coordinates failed to map
    final_df = final_df.dropna(subset=['Latitude', 'Longitude'])

    # Save to Google Drive
    output_path = f"{output_dir}/external_poi_raw.csv"
    final_df.to_csv(output_path, index=False)

    print("\n" + "="*60)
    print(f"🎉 PIPELINE SUCCESS: {len(final_df)} POIs securely extracted.")
    print(f"📁 Bronze data materialized correctly at: {output_path}")

Mounted at /content/drive
 Initiating OSMnx Production Ingestion Pipeline...
⏳ Securely downloading features for Western Province, Sri Lanka...
✅ Extracted 2995 POIs from Western Province, Sri Lanka.
⏳ Securely downloading features for Central Province, Sri Lanka...
✅ Extracted 1010 POIs from Central Province, Sri Lanka.
⏳ Securely downloading features for Southern Province, Sri Lanka...
✅ Extracted 1025 POIs from Southern Province, Sri Lanka.
⏳ Securely downloading features for North Western Province, Sri Lanka...
✅ Extracted 636 POIs from North Western Province, Sri Lanka.

🎉 PIPELINE SUCCESS: 5666 POIs securely extracted.
📁 Bronze data materialized correctly at: /content/drive/MyDrive/DataStorm_TeamZypher/lakehouse/bronze/external_poi_raw.csv
